In [ ]:
import os, re, datetime, glob
import pandas as pd
import numpy as np
from datetime import date
import datetime
from pathlib import Path


'''
Defines folders.
The variable "d" captures the current folder where all programs are saved.
/source/ is where raw data is stored
/data/ is where the clean dataset is stored
folder_spec saves downloaded files (pdfs)
'''
d = f"{os.getcwd()}/"
PATH_RAW = os.path.join(d, "..", "raw")   # raw inputs (downloads / intermediate CSVs)
PATH_DATA = os.path.join(d, "..", "data")    # processed modeling tables
# Output folder for model results (forecasts, performance tables)
PATH_OUTPUT = os.path.join(d, "..", "output")
# Create folder if it does not exist
os.makedirs(PATH_OUTPUT, exist_ok=True)
data_file = "data.csv"
meta_file = "meta.csv"


csv_files = glob.glob(os.path.join(PATH_RAW, "*.csv"))
to_remove = [ 'meta' ]
csv_files = [f for f in csv_files if to_remove[0] not in f]

meta = pd.read_csv(os.path.join(PATH_DATA, meta_file))
#meta = meta[['ticket' , 'series' , 'nominal', 'sa', 'unstructured', 'months_lag', 'agg_method', 'description']]
meta.drop(columns= ["freq", "start_date", "end_date", "obs"] , inplace=True)

# Build NTL

In [ ]:
# -------------------------------------------
# Nighttime Lights

# -------------------------------------------
file_name = [ f for f in csv_files if "ntl" in f and "data" in f ][0].split('\\')[-1]
dfi = pd.read_csv(f"{PATH_RAW}/{file_name}", parse_dates=True)
dfi = dfi.dropna(how='all', axis=1)
dfi.rename(columns = { 'JD' : 'date' }, inplace=True)
dfi.set_index('date', inplace=True)


data = pd.DataFrame()

dfntl = dfi
threshold3 = dfntl['DNBvalue3'].quantile(0.90)
sub3 = dfntl[dfntl['DNBvalue3']<threshold3]['DNBvalue3'].mean()
window = 30
value = 'DNBvalue3'


# ---------------------------------------------

# BY CATEGORY: 30-day window mean
dfntl = dfi
dfntl[value] = dfntl[value].apply(lambda x: sub3 if x > threshold3 else x)
dfntl = pd.DataFrame(dfntl.groupby( [ dfntl.index, 'sector'] )[value].mean()).reset_index()
dfntl = dfntl.set_index('date')
dfntl = dfntl.pivot_table(index=dfntl.index, columns='sector', values=value)
dfntl.index = pd.to_datetime(dfntl.index)
dfntl = dfntl.rolling(window, min_periods=1).mean().resample('MS').last()
dfntl.columns = [ f"ntl_{col.lower()}_meanw{window}" for col in dfntl.columns ]
dfntl = dfntl.resample('MS').last()
data = pd.concat( [ data , dfntl ] , axis=1 )

# Monthly mean of the daily mean
dfntl = dfi
dfntl[value] = dfntl[value].apply(lambda x: sub3 if x > threshold3 else x)
dfntl = pd.DataFrame(dfntl.groupby( [ dfntl.index] )[value].mean()).reset_index()
dfntl = dfntl.set_index('date')
dfntl.index = pd.to_datetime(dfntl.index)
dfntl = dfntl.rolling(window, min_periods=1).mean().resample('MS').last()
dfntl.columns = [ f"ntl_meanw{window}" for col in dfntl.columns ]
dfntl = dfntl.resample('MS').last()
data = pd.concat( [ data , dfntl ] , axis=1 )

# BY CATEGORY: mean
dfntl = dfi
dfntl[value] = dfntl[value].apply(lambda x: sub3 if x > threshold3 else x)
dfntl = pd.DataFrame(dfntl.groupby( [ dfntl.index, 'sector'] )[value].mean()).reset_index()
dfntl = dfntl.set_index('date')
dfntl = dfntl.pivot_table(index=dfntl.index, columns='sector', values=value)
dfntl.index = pd.to_datetime(dfntl.index)
dfntl = dfntl.resample('MS').mean()
dfntl.columns = [ f"ntl_{col.lower()}_mean" for col in dfntl.columns ]
dfntl = dfntl.resample('MS').last()
data = pd.concat( [ data , dfntl ] , axis=1 )

# Monthly mean of the daily mean
dfntl = dfi
dfntl[value] = dfntl[value].apply(lambda x: sub3 if x > threshold3 else x)
dfntl = pd.DataFrame(dfntl.groupby( [ dfntl.index] )[value].mean()).reset_index()
dfntl = dfntl.set_index('date')
dfntl.index = pd.to_datetime(dfntl.index)
dfntl = dfntl.resample('MS').mean()
dfntl.columns = [ f"ntl_mean" for col in dfntl.columns ]
dfntl = dfntl.resample('MS').last()
data = pd.concat( [ data , dfntl ] , axis=1 )

data.columns = (
    data.columns
      .str.replace(r"[&]", "", regex=True)        # remove &
      .str.replace(r"\s+", "_", regex=True)       # spaces → _
)
data.columns


# Main data

In [ ]:
def infer_freq_from_index(dt_index: pd.DatetimeIndex) -> str:
    dti = pd.DatetimeIndex(dt_index).dropna().sort_values().unique()
    if len(dti) < 3:
        return "unknown"

    deltas = pd.Series(dti[1:]) - pd.Series(dti[:-1])
    median_days = float(deltas.median() / pd.Timedelta(days=1))

    if median_days <= 2:
        return "daily"
    if 25 <= median_days <= 35:
        return "monthly"
    if 80 <= median_days <= 100:
        return "quarterly"

    # NEW: yearly (allow some wiggle for leap years / reporting conventions)
    if 320 <= median_days <= 430:
        return "yearly"

    return "unknown"


def daily_to_monthly_start(s: pd.Series, how: str = "last") -> pd.Series:
    if how == "last":
        return s.resample("MS").last()
    if how == "mean":
        return s.resample("MS").mean()
    if how == "sum":
        return s.resample("MS").sum()
    raise ValueError("how must be one of {'last','mean','sum'}")


def to_monthly_start(
    df: pd.DataFrame,
    freq_detected: str,
    agg_daily: str = "last",
    yearly_to_monthly: str = "jan",  # "jan" | "ffill" | "mean"
) -> pd.DataFrame:
    """
    Convert data at various frequencies into a DataFrame indexed at month-start ("MS").

    yearly_to_monthly:
      - "jan": place each yearly value on Jan 1 of that year (single observation per year).
      - "ffill": place on Jan 1 and forward-fill through the year's months.
      - "mean": spread yearly value across all months (same as ffill for levels, but explicit).
    """
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    if freq_detected == "daily":
        return df.apply(lambda col: daily_to_monthly_start(col, how=agg_daily))

    if freq_detected == "monthly":
        m = df.groupby(df.index.to_period("M")).last()
        m.index = m.index.to_timestamp(how="start")
        return m

    if freq_detected == "quarterly":
        q = df.groupby(df.index.to_period("Q")).last()
        # map quarter to the month-start of the quarter end month (your existing convention)
        q.index = q.index.to_timestamp(how="end").to_period("M").to_timestamp(how="start")
        return q

    # NEW: yearly
    if freq_detected == "yearly":
        y = df.groupby(df.index.to_period("Y")).last()
        y.index = y.index.to_timestamp(how="start")  # Jan 1

        if yearly_to_monthly == "jan":
            return y

        # Expand to monthly-start index for the span covered by the data
        monthly_index = pd.date_range(
            start=y.index.min(),
            end=(y.index.max() + pd.offsets.YearEnd(0)).to_timestamp() if hasattr(y.index.max(), "to_timestamp") else (y.index.max() + pd.offsets.YearEnd(0)),
            freq="MS",
        )

        y = y.reindex(monthly_index)

        if yearly_to_monthly in {"ffill", "mean"}:
            # Fill each year across its months
            return y.groupby(y.index.year).ffill()

        raise ValueError("yearly_to_monthly must be one of {'jan','ffill','mean'}")

    # fallback: collapse to monthly start anyway
    m = df.groupby(df.index.to_period("M")).last()
    m.index = m.index.to_timestamp(how="start")
    return m


def read_one_csv(path: str | Path, date_col: str = "date") -> pd.DataFrame:
    df = pd.read_csv(path)
    if date_col not in df.columns:
        date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df = df[~df.index.duplicated(keep="last")]
    return df


def series_metadata(df: pd.DataFrame, freq_detected: str, source_file: str) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        nonmiss = s.dropna()
        if nonmiss.empty:
            start = pd.NaT
            end = pd.NaT
            n = 0
        else:
            start = nonmiss.index.min()
            end = nonmiss.index.max()
            n = int(nonmiss.shape[0])

        rows.append({
            "series": col,
            "file": source_file,
            "freq": freq_detected,
            "start_date": start,
            "end_date": end,
            "obs": n,
        })
    return pd.DataFrame(rows)


def merge_csv_files_with_metadata(
    csv_files,
    date_col="Date",
    agg_daily="last",
    prefix_cols=False,
    yearly_to_monthly="jan",  # NEW passthrough
):
    merged = None
    meta_list = []

    for f in csv_files:
        f = str(f)
        raw = read_one_csv(f, date_col=date_col)

        if "blk_ntl" in f:
            try:
                raw = raw.drop(columns="DNB_BRDF-Corrected_NTL")
            except Exception:
                pass
        if "gtrends" in f:
            raw["gtrends_mean"] = raw.mean(axis=1)
            raw.index = pd.to_datetime(raw.index)

        raw_freq = infer_freq_from_index(raw.index)

        df = raw.copy()
        if prefix_cols:
            stem = Path(f).stem
            df = df.add_prefix(stem + "__")

        df = to_monthly_start(df, raw_freq, agg_daily=agg_daily, yearly_to_monthly=yearly_to_monthly)

        # Drop columns already present in merged (duplicates across source files)
        if merged is not None:
            dup_cols = [c for c in df.columns if c in merged.columns]
            if dup_cols:
                print(f"[dedup] {Path(f).name}: dropping {dup_cols} — already loaded from a prior file")
                df = df.drop(columns=dup_cols)

        meta_list.append(series_metadata(df, raw_freq, source_file=Path(f).name))
        merged = df if merged is None else merged.join(df, how="outer")

    merged = merged.sort_index()
    metadata = pd.concat(meta_list, ignore_index=True)

    metadata["start_date"] = pd.to_datetime(metadata["start_date"]).dt.date
    metadata["end_date"] = pd.to_datetime(metadata["end_date"]).dt.date

    return merged, metadata


# usage:
csv_files = [f for f in csv_files if "ntl_data" not in f]
df, meta_df = merge_csv_files_with_metadata(
    csv_files,
    date_col="date",
    agg_daily="last",
    yearly_to_monthly="jan",  # or "ffill"
)
meta_df = meta_df[["series", "freq", "start_date", "end_date", "obs"]]

In [ ]:
[ col for col in df.columns if "Nadir" in col ]

# Metadata

In [ ]:
meta_df = pd.concat( [meta_df , series_metadata(data, "daily", source_file="ntl_data.csv")[["series", "freq", "start_date", "end_date", "obs"]] ], axis=0).reset_index(drop=True)

meta_df['start_date'] = pd.to_datetime(meta_df['start_date'])
meta_df['end_date'] = pd.to_datetime(meta_df['end_date'])

df = df.join(data, how="outer")

In [ ]:
#merged_meta = meta.drop(columns = ['start_date', 'end_date', 'freq', 'obs']).merge(meta_df, on="series", how="outer")
merged_meta = meta.merge(meta_df, on="series", how="outer")

# Ensure datetime
merged_meta["end_date"] = pd.to_datetime(merged_meta["end_date"])

# Ensure column exists
if "months_lag" not in merged_meta.columns:
    merged_meta["months_lag"] = pd.NA

# Current month (year + month only)
today = pd.Timestamp.today()
current_year = today.year
current_month = today.month

mask = merged_meta["months_lag"].isna() & merged_meta["end_date"].notna()

merged_meta.loc[mask, "months_lag"] = (
    (current_year - merged_meta.loc[mask, "end_date"].dt.year) * 12
    + (current_month - merged_meta.loc[mask, "end_date"].dt.month)
).astype("Int64")   # nullable integer dtype

# months_lag: float -> nullable int
merged_meta["months_lag"] = pd.to_numeric(merged_meta["months_lag"], errors="coerce").round().astype("Int64")

# obs: float -> nullable int
merged_meta["obs"] = pd.to_numeric(merged_meta["obs"], errors="coerce").round().astype("Int64")
merged_meta["nominal"] = pd.to_numeric(merged_meta["nominal"], errors="coerce").round().astype("Int64")
merged_meta["sa"] = pd.to_numeric(merged_meta["sa"], errors="coerce").round().astype("Int64")
merged_meta

In [ ]:
# Save metadata

merged_meta.to_csv(os.path.join(PATH_DATA, meta_file), index=False)

## Rename Columns

In [ ]:
rename_dict = dict(
    zip(merged_meta["series"], merged_meta["ticket"])
)
# Only rename where ticket is a non-null string; leave unregistered series under their original name
rename_dict = {k: v for k, v in rename_dict.items() if isinstance(v, str) and v == v}
df = df.rename(columns=rename_dict)

# Drop any columns with NaN names (series not yet registered in meta.csv)
nan_cols = [col for col in df.columns if not isinstance(col, str)]
if nan_cols:
    print(f"[unregistered] dropping {len(nan_cols)} column(s) with no ticket in meta.csv")
df = df.drop(columns=nan_cols)

first_gdp = df['RGDP0000'].dropna().index.min()
# subtract 2 months
cutoff = (first_gdp.to_period("M") - 2).to_timestamp(how="start")

df = df[df.index >= cutoff]

df.index.name = 'date'

df.drop( columns = [col for col in df.columns if isinstance(col, str) and "RGDP" in col and "RGDP0000" not in col], inplace=True)
df.drop( columns = [col for col in df.columns if isinstance(col, str) and "RCPI" in col and "RCPI0000" not in col], inplace=True)
df = df[["RGDP0000"] + [col for col in df.columns if "RGDP0000" not in str(col)]]

df = df.apply(
    lambda col: pd.to_numeric(
        col.astype(str)
           .str.replace(",", "", regex=False)
           .str.replace("\n", "", regex=False)
           .str.strip(),
        errors="coerce"
    )
)

#df = df[df.index >= "2014-01-01"]

print("Data range from {} to {}".format(df.index.min().date(), df.index.max().date()))
print("GDP range from {} to {}".format(df['RGDP0000'].dropna().index.min().date(), df['RGDP0000'].dropna().index.max().date()))

In [ ]:
df.drop( columns = [col for col in df.columns if "RGDP" in col and "RGDP0000" not in col], inplace=True)
df.drop( columns = [col for col in df.columns if "RCPI" in col and "RCPI0000" not in col], inplace=True)
df = df[["RGDP0000"] + [col for col in df.columns if "RGDP0000" not in col ]]

print("Data range from {} to {}".format(df.index.min().date(), df.index.max().date()))
print("GDP range from {} to {}".format(df['RGDP0000'].dropna().index.min().date(), df['RGDP0000'].dropna().index.max().date()))

In [ ]:
df.to_csv(os.path.join(PATH_DATA, data_file), index=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(11, 4))

s = df['RGDP0000'].dropna()

ax.fill_between(s.index, s, alpha=0.15, color='steelblue')
ax.plot(s.index, s, color='steelblue', linewidth=2)

ax.set_title("Trinidad & Tobago — Real GDP", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("TT$ millions (current prices)", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.7)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

s = df['UNTL3000'].dropna()

ax.fill_between(s.index, s, alpha=0.15, color='goldenrod')
ax.plot(s.index, s, color='goldenrod', linewidth=2)

ax.set_title("Trinidad & Tobago — Nighttime Lights (Overall Mean)", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("DNB radiance (nW·cm⁻²·sr⁻¹)", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.7)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()